In [1]:
import os, csv
import pandas as pd

os.getcwd()

'/Users/z5611110/Documents/GitHub/MLLM-MSR/MLLM-MSR/data/microlens'

In [2]:
df = pd.read_csv('../microlens/processed/MicroLens-50k_pairs.csv')
print(f'shape: {df.shape}')
df[:5]

shape: (359708, 3)


,user,item,timestamp
0,36121,9580,1583378629552
1,26572,9580,1583436719018
2,37550,9580,1584412681021
3,14601,9580,1584848802432
4,15061,9580,1585388171106


In [3]:
from collections import Counter
import numpy as np

min_u_num, min_i_num = 7, 5

def get_illegal_ids_by_inter_num(df, field, max_num=None, min_num=None):
    if field is None:
        return set()
    if max_num is None and min_num is None:
        return set()

    max_num = max_num or np.inf
    min_num = min_num or -1

    ids = df[field].values
    inter_num = Counter(ids)
    ids = {id_ for id_ in inter_num if inter_num[id_] < min_num or inter_num[id_] > max_num}
    print(f'{len(ids)} illegal_ids_by_inter_num, field={field}')

    return ids


def filter_by_k_core(df):
    while True:
        ban_users = get_illegal_ids_by_inter_num(df, field='user', max_num=None, min_num=min_u_num)
        ban_items = get_illegal_ids_by_inter_num(df, field='item', max_num=None, min_num=min_i_num)
        if len(ban_users) == 0 and len(ban_items) == 0:
            return

        dropped_inter = pd.Series(False, index=df.index)
        if 'user':
            dropped_inter |= df['user'].isin(ban_users)
        if 'item':
            dropped_inter |= df['item'].isin(ban_items)
        print(f'{len(dropped_inter)} dropped interactions')
        df.drop(df.index[dropped_inter], inplace=True)

In [4]:
filter_by_k_core(df)
print(f'k-core shape: {df.shape}')
print(f'shape after k-core: {df.shape}')
df[:5]

31090 illegal_ids_by_inter_num, field=user
4552 illegal_ids_by_inter_num, field=item
359708 dropped interactions
1211 illegal_ids_by_inter_num, field=user
3394 illegal_ids_by_inter_num, field=item
187346 dropped interactions
2032 illegal_ids_by_inter_num, field=user
434 illegal_ids_by_inter_num, field=item
171243 dropped interactions
294 illegal_ids_by_inter_num, field=user
684 illegal_ids_by_inter_num, field=item
158350 dropped interactions
581 illegal_ids_by_inter_num, field=user
120 illegal_ids_by_inter_num, field=item
154171 dropped interactions
94 illegal_ids_by_inter_num, field=user
222 illegal_ids_by_inter_num, field=item
150312 dropped interactions
179 illegal_ids_by_inter_num, field=user
41 illegal_ids_by_inter_num, field=item
148903 dropped interactions
36 illegal_ids_by_inter_num, field=user
76 illegal_ids_by_inter_num, field=item
147681 dropped interactions
73 illegal_ids_by_inter_num, field=user
16 illegal_ids_by_inter_num, field=item
147166 dropped interactions
12 illegal

,user,item,timestamp
0,36121,9580,1583378629552
2,37550,9580,1584412681021
3,14601,9580,1584848802432
4,15061,9580,1585388171106
6,3542,9580,1585404918503


In [5]:
print(df['user'].nunique())
print(df['item'].nunique())

14323
9535


In [6]:
import pandas as pd
import numpy as np

# df has three columns: 'user', 'item', 'timestamp'

# calculate the frequency of each item
item_frequency = df.groupby('item').size().reset_index(name='frequency')

# sort items by frequency in descending order
item_frequency_sorted = item_frequency.sort_values(by='frequency', ascending=False)

# sort the original dataframe by timestamp
df_sorted = df.sort_values('timestamp')

# get all unique items and users
all_items = set(df_sorted['item'].unique())
all_users = df_sorted['user'].unique()

# generate negative samples for each user
np.random.seed(42)
negative_samples_per_user = {}
for user in all_users:
    user_items = set(df_sorted[df_sorted['user'] == user]['item'].unique())
    available_items = sorted(all_items - user_items)
    negative_samples = np.random.choice(available_items, size=min(20, len(available_items)), replace=False)
    negative_samples_per_user[user] = negative_samples

print(len(negative_samples_per_user.keys()))

14323


In [7]:
top_6_items_per_user = {}
for user in all_users:
    # get the items interacted by the user
    user_items = df_sorted[df_sorted['user'] == user]['item']
    # get the top 6 items based on frequency
    top_6_items = user_items.map(item_frequency_sorted.set_index('item')['frequency']).sort_values(ascending=False).index[:11]
    # sorted these top 6 items by timestamp
    top_6_items_per_user[user] = df_sorted.loc[top_6_items].sort_values('timestamp')['item'].values

In [8]:
lines = []
for user in all_users:
    top_items_str = ', '.join([str(item) for item in top_6_items_per_user[user]])
    negative_samples_str = ', '.join([str(item) for item in negative_samples_per_user[user]])
    # format the line as "user_id\ttop_items\tnegative_samples"
    line = f"{user}\t{top_items_str}\t{negative_samples_str}"
    lines.append(line)

tsv_file_path = '../microlens/processed/user_items_negs.tsv'
with open(tsv_file_path, 'w') as file:
    file.write('\n'.join(lines))

In [9]:
# full chronological history per user (no top-k cap) for path-guided history selection
full_history_lines = []
for user in all_users:
    user_items = df_sorted[df_sorted["user"] == user].sort_values("timestamp")["item"].values
    items_str = ",".join(str(item) for item in user_items)
    full_history_lines.append(str(user) + "\t" + items_str + "\n")

full_hist_path = "../microlens/processed/user_full_history.tsv"
with open(full_hist_path, "w") as f:
    f.writelines(full_history_lines)
print(f"Saved {full_hist_path} with {len(full_history_lines)} users")

Saved ../microlens/processed/user_full_history.tsv with 14323 users


In [10]:
from sklearn.model_selection import train_test_split

tsv_file_path = '../microlens/processed/user_items_negs.tsv'
data = pd.read_csv(tsv_file_path, sep='\t', header=None, names=['user', 'items', 'negative_samples'])

users = data['user'].unique()

train_users, test_val_users = train_test_split(users, test_size=0.2, random_state=42)

val_users, test_users = train_test_split(test_val_users, test_size=0.5, random_state=42)

train_data = data[data['user'].isin(train_users)]
val_data = data[data['user'].isin(val_users)]
test_data = data[data['user'].isin(test_users)]

print(f"Training data size: {len(train_data)}")
print(f"Validation data size: {len(val_data)}")
print(f"Test data size: {len(test_data)}")

Training data size: 11458
Validation data size: 1432
Test data size: 1433


In [11]:
def save_to_tsv(df, file_path):
    df.to_csv(file_path, sep='\t', header=False, index=False)
    print(f"File saved to {file_path}")

# save the dataframes to TSV files
save_to_tsv(train_data, '../microlens/processed/train.tsv')
save_to_tsv(val_data, '../microlens/processed/val.tsv')
save_to_tsv(test_data, '../microlens/processed/test.tsv')

File saved to ../microlens/processed/train.tsv
File saved to ../microlens/processed/val.tsv
File saved to ../microlens/processed/test.tsv
